# 00 — Carga y validación de datos

**Grupo 10 — Minería de Datos**

Este notebook carga los 3 datasets del MINEDUC, valida su integridad,
realiza el join por MRUN y guarda los datasets procesados en `data/processed/`.

Solo necesita ejecutarse una vez. Los demás notebooks cargan desde `data/processed/`.

## Datasets
| ID | Archivo | Descripción | Filas esperadas |
|----|---------|--------------|-----------------|
| A | `A_Puntajes.csv` | Inscritos PAES 2025 + puntajes | ~313.759 |
| B | `B_Socioeconomico.csv` | Domicilio y datos socioeconómicos | ~313.759 |
| C | `C_Matricula.csv` | Matrícula Ed. Superior 2025 | ~1.455.639 |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─── Rutas ──────────────────────────────────────────────────────────────────
# Detectar automáticamente la raíz del proyecto buscando la carpeta data/
def encontrar_raiz():
    """Sube directorios hasta encontrar la carpeta data/raw/"""
    carpeta = Path.cwd()
    for _ in range(5):  # máximo 5 niveles hacia arriba
        if (carpeta / 'data' / 'raw').exists():
            return carpeta
        carpeta = carpeta.parent
    raise FileNotFoundError(
        f"No se encontró data/raw/ desde {Path.cwd()}.\n"
        "Asegúrate de que los CSVs estén en data/raw/ relativo a la raíz del repo."
    )

RAIZ = encontrar_raiz()
DATA_RAW  = RAIZ / 'data' / 'raw'
DATA_PROC = RAIZ / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f'Directorio actual:  {Path.cwd()}')
print(f'Raíz del proyecto:  {RAIZ}')
print(f'RAW:                {DATA_RAW}')
print(f'PROCESSED:          {DATA_PROC}')
print()
print('Archivos en data/raw/:')
for f in sorted(DATA_RAW.iterdir()):
    size = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}  ({size:.0f} MB)')

Directorio actual:  g:\Mi unidad\Universidad\DCC\2do Año\Mineria de Datos\Hito1\Hito2
Raíz del proyecto:  g:\Mi unidad\Universidad\DCC\2do Año\Mineria de Datos\Hito1
RAW:                g:\Mi unidad\Universidad\DCC\2do Año\Mineria de Datos\Hito1\data\raw
PROCESSED:          g:\Mi unidad\Universidad\DCC\2do Año\Mineria de Datos\Hito1\data\processed

Archivos en data/raw/:
  A_Puntajes.csv  (223 MB)
  B_Socioeconomico.csv  (14 MB)
  C_Matricula.csv  (903 MB)
  README.txt  (0 MB)


## 1. Carga de datasets

In [2]:
# ─── Dataset A: Puntajes PAES ────────────────────────────────────────────────
# Separador: punto y coma (;)
COLS_A = [
    'MRUN', 'RAMA_EDUCACIONAL', 'DEPENDENCIA', 'CODIGO_REGION_EGRESO',
    'ANYO_DE_EGRESO', 'PTJE_NEM', 'PTJE_RANKING',
    'CLEC_MAX', 'MATE1_MAX', 'MATE2_MAX', 'HCSOC_MAX', 'CIEN_MAX',
    'CORRECTAS_REG_CL', 'ERRADAS_REG_CL', 'OMITIDAS_REG_CL',
    'CORRECTAS_REG_M1', 'ERRADAS_REG_M1', 'OMITIDAS_REG_M1',
]

print('Cargando Dataset A (puntajes)...')
df_A = pd.read_csv(
    DATA_RAW / 'A_Puntajes.csv',
    sep=';',
    usecols=lambda c: c in COLS_A,
    encoding='utf-8',
    low_memory=False
)
print(f'  Filas: {len(df_A):,} | Columnas: {df_A.shape[1]}')
print(f'  Columnas cargadas: {df_A.columns.tolist()}')
df_A.head(3)

Cargando Dataset A (puntajes)...
  Filas: 313,759 | Columnas: 18
  Columnas cargadas: ['MRUN', 'RAMA_EDUCACIONAL', 'DEPENDENCIA', 'CODIGO_REGION_EGRESO', 'ANYO_DE_EGRESO', 'PTJE_NEM', 'PTJE_RANKING', 'CLEC_MAX', 'MATE1_MAX', 'MATE2_MAX', 'HCSOC_MAX', 'CIEN_MAX', 'CORRECTAS_REG_CL', 'ERRADAS_REG_CL', 'OMITIDAS_REG_CL', 'CORRECTAS_REG_M1', 'ERRADAS_REG_M1', 'OMITIDAS_REG_M1']


,MRUN,RAMA_EDUCACIONAL,DEPENDENCIA,CODIGO_REGION_EGRESO,ANYO_DE_EGRESO,PTJE_NEM,PTJE_RANKING,CLEC_MAX,MATE1_MAX,MATE2_MAX,HCSOC_MAX,CIEN_MAX,CORRECTAS_REG_CL,ERRADAS_REG_CL,OMITIDAS_REG_CL,CORRECTAS_REG_M1,ERRADAS_REG_M1,OMITIDAS_REG_M1
0,214,T2,5,16,2024,418,418,0,0,0,0,0,0,0,0,0,0,0
1,492,T3,6,12,2023,609,609,617,557,0,415,0,34,26,0,28,32,0
2,534,T1,2,6,2024,752,757,494,662,308,0,455,22,38,0,38,22,0


In [3]:
# ─── Dataset B: Socioeconómico ───────────────────────────────────────────────
# Separador: punto y coma (;)
COLS_B = [
    'MRUN', 'CODIGO_REGION_DOMICILIO', 'CODIGO_PROVINCIA_DOMICILIO',
    'CODIGO_COMUNA_DOMICILIO', 'NOMBRE_COMUNA_DOMICILIO',
    'SEXO', 'INGRESO_PERCAPITA_GRUPO_FA', 'RAZON_PRINCIPAL_PAES',
]

print('Cargando Dataset B (socioeconómico)...')
df_B = pd.read_csv(
    DATA_RAW / 'B_Socioeconomico.csv',
    sep=';',
    usecols=lambda c: c in COLS_B,
    encoding='utf-8',
    low_memory=False
)
print(f'  Filas: {len(df_B):,} | Columnas: {df_B.shape[1]}')
print(f'  Columnas cargadas: {df_B.columns.tolist()}')
df_B.head(3)

Cargando Dataset B (socioeconómico)...
  Filas: 313,759 | Columnas: 8
  Columnas cargadas: ['MRUN', 'CODIGO_REGION_DOMICILIO', 'CODIGO_PROVINCIA_DOMICILIO', 'CODIGO_COMUNA_DOMICILIO', 'NOMBRE_COMUNA_DOMICILIO', 'SEXO', 'INGRESO_PERCAPITA_GRUPO_FA', 'RAZON_PRINCIPAL_PAES']


,MRUN,CODIGO_REGION_DOMICILIO,CODIGO_PROVINCIA_DOMICILIO,CODIGO_COMUNA_DOMICILIO,NOMBRE_COMUNA_DOMICILIO,SEXO,INGRESO_PERCAPITA_GRUPO_FA,RAZON_PRINCIPAL_PAES
0,214,16,161,16101,CHILLAN,1,1,3
1,492,12,121,12101,PUNTA ARENAS,2,9,4
2,534,6,63,6305,NANCAGUA,2,2,3


In [4]:
# ─── Dataset C: Matrícula ────────────────────────────────────────────────────
# Separador: punto y coma (;) | Columnas en minúsculas → se normalizan a mayúsculas
COLS_C_lower = [
    'mrun', 'cat_periodo', 'tipo_inst_1', 'tipo_inst_2',
    'nomb_inst', 'nomb_carrera', 'area_conocimiento',
    'region_sede', 'forma_ingreso', 'valor_arancel',
    'acreditada_inst', 'nivel_global', 'nivel_carrera_2',
]

print('Cargando Dataset C (matrícula)...')
df_C = pd.read_csv(
    DATA_RAW / 'C_Matricula.csv',
    sep=';',
    usecols=lambda c: c.lower() in COLS_C_lower,
    encoding='utf-8',
    low_memory=False
)

# Normalizar nombres de columnas a MAYÚSCULAS para consistencia con A y B
df_C.columns = df_C.columns.str.upper()

# Filtrar solo matrículas de 2025
if 'CAT_PERIODO' in df_C.columns:
    df_C = df_C[df_C['CAT_PERIODO'] == 2025]

print(f'  Filas: {len(df_C):,} | Columnas: {df_C.shape[1]}')
print(f'  Columnas cargadas: {df_C.columns.tolist()}')
df_C.head(3)

Cargando Dataset C (matrícula)...
  Filas: 1,455,612 | Columnas: 13
  Columnas cargadas: ['CAT_PERIODO', 'MRUN', 'TIPO_INST_1', 'TIPO_INST_2', 'NOMB_INST', 'NOMB_CARRERA', 'REGION_SEDE', 'NIVEL_GLOBAL', 'NIVEL_CARRERA_2', 'VALOR_ARANCEL', 'AREA_CONOCIMIENTO', 'ACREDITADA_INST', 'FORMA_INGRESO']


,CAT_PERIODO,MRUN,TIPO_INST_1,TIPO_INST_2,NOMB_INST,NOMB_CARRERA,REGION_SEDE,NIVEL_GLOBAL,NIVEL_CARRERA_2,VALOR_ARANCEL,AREA_CONOCIMIENTO,ACREDITADA_INST,FORMA_INGRESO
0,2025,5.0,Institutos Profesionales,Institutos Profesionales,IP LATINOAMERICANO DE COMERCIO EXTERIOR - IPLACEX,INGENIERIA EN PREVENCION DE RIESGOS,Metropolitana,Pregrado,Carreras Profesionales,2143000.0,Tecnología,ACREDITADA,1
1,2025,19.0,Centros de Formación Técnica,Centros de Formación Técnica,CFT INACAP,ADMINISTRACION DE EMPRESAS,Arica y Parinacota,Pregrado,Carreras Técnicas,2601000.0,Administración y Comercio,ACREDITADA,1
2,2025,26.0,Universidades,Universidades Privadas,UNIVERSIDAD DEL DESARROLLO,DERECHO,Metropolitana,Pregrado,Carreras Profesionales,232.0,Derecho,ACREDITADA,1


## 2. Validación básica

In [5]:
def validar_dataset(df, nombre):
    print(f'\n{'='*50}')
    print(f'Dataset {nombre}: {len(df):,} filas x {df.shape[1]} columnas')
    print(f'MRUNs únicos: {df["MRUN"].nunique():,}')
    print(f'MRUNs duplicados: {df["MRUN"].duplicated().sum():,}')
    
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(f'Columnas con nulos:')
        for col, n in nulos.items():
            print(f'  {col}: {n:,} ({n/len(df)*100:.1f}%)')
    else:
        print('Sin valores nulos')

validar_dataset(df_A, 'A (puntajes)')
validar_dataset(df_B, 'B (socioeconómico)')
validar_dataset(df_C, 'C (matrícula 2025)')


Dataset A (puntajes): 313,759 filas x 18 columnas
MRUNs únicos: 313,759
MRUNs duplicados: 0
Columnas con nulos:
  RAMA_EDUCACIONAL: 2,434 (0.8%)

Dataset B (socioeconómico): 313,759 filas x 8 columnas
MRUNs únicos: 313,759
MRUNs duplicados: 0
Sin valores nulos

Dataset C (matrícula 2025): 1,455,612 filas x 13 columnas
MRUNs únicos: 1,436,931
MRUNs duplicados: 18,680
Columnas con nulos:
  MRUN: 2,580 (0.2%)


## 3. Join A + B → base para P1 y P2

In [6]:
print('Join A + B por MRUN...')
df_base = df_A.merge(df_B, on='MRUN', how='inner')
print(f'Antes del join: A={len(df_A):,} | B={len(df_B):,}')
print(f'Después del join: {len(df_base):,} filas')
print(f'Pérdida: {len(df_A) - len(df_base):,} registros sin match')

Join A + B por MRUN...
Antes del join: A=313,759 | B=313,759
Después del join: 313,759 filas
Pérdida: 0 registros sin match


In [7]:
# ─── Filtro 1: Excluir quienes no rindieron la PAES ─────────────────────────
# Puntaje 0 significa que se inscribió pero no rindió
n_antes = len(df_base)
df_base = df_base[df_base['CLEC_MAX'] > 0]
print(f'Filtro puntaje > 0: eliminados {n_antes - len(df_base):,} registros')

# ─── Filtro 2: Excluir decil inválido (99 = prefiero no responder) ───────────
n_antes = len(df_base)
df_base = df_base[df_base['INGRESO_PERCAPITA_GRUPO_FA'] != 99]
print(f'Filtro decil válido: eliminados {n_antes - len(df_base):,} registros')

print(f'\nDataset base final: {len(df_base):,} estudiantes válidos')

Filtro puntaje > 0: eliminados 38,067 registros
Filtro decil válido: eliminados 75,595 registros

Dataset base final: 200,097 estudiantes válidos


## 4. Join base + C → dataset completo para P3 y P4

In [8]:
print('Join base + C por MRUN...')
df_completo = df_base.merge(df_C, on='MRUN', how='inner')
print(f'Base: {len(df_base):,} | Con matrícula: {len(df_completo):,}')
print(f'Estudiantes que NO se matricularon: {len(df_base) - len(df_completo):,}')
print(f'Tasa de matriculación: {len(df_completo)/len(df_base)*100:.1f}%')

Join base + C por MRUN...
Base: 200,097 | Con matrícula: 136,156
Estudiantes que NO se matricularon: 63,941
Tasa de matriculación: 68.0%


## 5. Construcción de variables derivadas

In [10]:
# ─── Variable: migra ─────────────────────────────────────────────────────────
# Mapeo de código de región a nombre (según Anexo I del dataset B)
REGION_MAP = {
    1: 'Región de Tarapacá',
    2: 'Región de Antofagasta',
    3: 'Región de Atacama',
    4: 'Región de Coquimbo',
    5: 'Región de Valparaíso',
    6: "Región del Libertador General Bernardo O'Higgins",
    7: 'Región del Maule',
    8: 'Región del Biobío',
    9: 'Región de La Araucanía',
    10: 'Región de Los Lagos',
    11: 'Región Aysén del General Carlos Ibáñez del Campo',
    12: 'Región de Magallanes y de la Antártica Chilena',
    13: 'Región Metropolitana de Santiago',
    14: 'Región de Los Ríos',
    15: 'Región de Arica y Parinacota',
    16: 'Región de Ñuble',
}

df_completo['region_domicilio_nombre'] = df_completo['CODIGO_REGION_DOMICILIO'].map(REGION_MAP)

# migra = 1 si la región de la institución es distinta a la de domicilio
# Nota: REGION_SEDE viene como nombre en el Dataset C
df_completo['migra'] = (
    df_completo['region_domicilio_nombre'].str.lower().str.strip() != 
    df_completo['REGION_SEDE'].str.lower().str.strip()
).astype(int)

print('Distribución variable migra:')
print(df_completo['migra'].value_counts())
print(f'\nTasa de migración: {df_completo["migra"].mean()*100:.1f}%')

Distribución variable migra:
migra
1    136156
Name: count, dtype: int64

Tasa de migración: 100.0%


In [ ]:
# ─── Variable: es_tecnico ────────────────────────────────────────────────────
df_base['es_tecnico'] = df_base['RAMA_EDUCACIONAL'].str.startswith('T').fillna(False).astype(int)
df_completo['es_tecnico'] = df_completo['RAMA_EDUCACIONAL'].str.startswith('T').fillna(False).astype(int)
print(f'es_tecnico — Técnicos: {df_base["es_tecnico"].sum():,} ({df_base["es_tecnico"].mean()*100:.1f}%)')

# ─── Variable: migra ─────────────────────────────────────────────────────────
# Mapa: nombre corto usado por SIES en REGION_SEDE → código de región
SEDE_A_CODIGO = {
    'Tarapacá':                1,
    'Antofagasta':             2,
    'Atacama':                 3,
    'Coquimbo':                4,
    'Valparaíso':              5,
    "O'Higgins":               6,
    "Lib. Gral. B. O'Higgins": 6,   # variante encontrada en el dataset
    'Maule':                   7,
    'Biobío':                  8,
    'Bío-Bío':                 8,
    'Araucanía':               9,
    'La Araucanía':            9,
    'Los Lagos':              10,
    'Aysén':                  11,
    'Magallanes':             12,
    'Metropolitana':          13,
    'Los Ríos':               14,
    'Arica y Parinacota':     15,
    'Ñuble':                  16,
}

# Mapear REGION_SEDE → código numérico
df_completo['codigo_region_sede'] = df_completo['REGION_SEDE'].map(SEDE_A_CODIGO)

# Verificar si quedaron valores sin mapear
sin_mapeo = df_completo['codigo_region_sede'].isna().sum()
if sin_mapeo > 0:
    print(f'\n⚠️  {sin_mapeo} registros con REGION_SEDE sin mapear:')
    print(df_completo[df_completo['codigo_region_sede'].isna()]['REGION_SEDE'].value_counts())
else:
    print('✅  Todos los valores de REGION_SEDE mapeados correctamente')

# migra = 1 si el código de la sede difiere del código de domicilio
df_completo['migra'] = (
    df_completo['codigo_region_sede'] != df_completo['CODIGO_REGION_DOMICILIO']
).astype(int)

print('\nDistribución variable migra:')
print(df_completo['migra'].value_counts())
print(f'Tasa de migración: {df_completo["migra"].mean()*100:.1f}%')

# Tasa por región de domicilio (top 5 y bottom 5)
tasa_region = df_completo.groupby('CODIGO_REGION_DOMICILIO')['migra'].mean().sort_values(ascending=False)
print('\nRegiones con mayor migración (top 5):')
print(tasa_region.head())
print('\nRegiones con menor migración (bottom 5):')
print(tasa_region.tail())

In [ ]:
import matplotlib.pyplot as plt

# ─── Limpieza previa de DEPENDENCIA ─────────────────────────────────────────
df_base['DEPENDENCIA'] = pd.to_numeric(df_base['DEPENDENCIA'], errors='coerce')
n_invalidos = df_base['DEPENDENCIA'].isna().sum()
if n_invalidos > 0:
    print(f'⚠️  {n_invalidos} filas con DEPENDENCIA inválida → eliminadas')
    df_base = df_base.dropna(subset=['DEPENDENCIA'])
    df_base['DEPENDENCIA'] = df_base['DEPENDENCIA'].astype(int)

# ─── Asegurar que es_tecnico existe ─────────────────────────────────────────
if 'es_tecnico' not in df_base.columns:
    df_base['es_tecnico'] = df_base['RAMA_EDUCACIONAL'].str.startswith('T').astype(int)
    print('ℹ️  es_tecnico creado en esta celda')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribución de variables clave — Dataset base', fontsize=14)

# Puntajes
df_base['CLEC_MAX'].hist(bins=50, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Puntaje CLEC (lenguaje)')

df_base['MATE1_MAX'].hist(bins=50, ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Puntaje MATE1 (matemática)')

df_base['PTJE_NEM'].hist(bins=50, ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Puntaje NEM')

# Decil de ingreso
df_base['INGRESO_PERCAPITA_GRUPO_FA'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,0], color='coral')
axes[1,0].set_title('Decil de ingreso familiar')
axes[1,0].set_xlabel('Decil')

# Dependencia — labels dinámicos
DEPENDENCIA_MAP = {1: 'Corp.Mun', 2: 'Municipal', 3: 'Part.Subv',
                   4: 'Part.Pag', 5: 'Corp.Del', 6: 'SLE'}
dep_counts = df_base['DEPENDENCIA'].value_counts().sort_index()
dep_counts.plot(kind='bar', ax=axes[1,1], color='coral')
axes[1,1].set_title('Dependencia del colegio')
dep_labels = [DEPENDENCIA_MAP.get(int(v), str(int(v))) for v in dep_counts.index]
axes[1,1].set_xticklabels(dep_labels, rotation=30)

# Rama educacional
df_base['es_tecnico'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,2], color='coral')
axes[1,2].set_title('Rama: Técnico vs Humanista')
axes[1,2].set_xticklabels(['Humanista', 'Técnico'], rotation=0)

plt.tight_layout()
plt.savefig(DATA_PROC / 'distribucion_variables.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nDataset base: {len(df_base):,} filas')
print(f'Valores únicos DEPENDENCIA: {sorted(df_base["DEPENDENCIA"].unique().tolist())}')
print(f'Técnico-profesionales: {df_base["es_tecnico"].sum():,} ({df_base["es_tecnico"].mean()*100:.1f}%)')

In [ ]:
# ─── Verificar colinealidad NEM / Ranking ────────────────────────────────────
corr = df_base[['PTJE_NEM', 'PTJE_RANKING', 'CLEC_MAX', 'MATE1_MAX']].corr()
print('Matriz de correlación (variables numéricas principales):')
print(corr.round(3))

corr_nem_ranking = corr.loc['PTJE_NEM', 'PTJE_RANKING']
print(f'\nCorrelación NEM ↔ Ranking: {corr_nem_ranking:.3f}')
if corr_nem_ranking > 0.85:
    print('⚠️  Alta correlación — considerar eliminar PTJE_NEM del clustering')
else:
    print('✅  Correlación aceptable — se pueden usar ambas')

## 7. Guardar datasets procesados

Usamos **Parquet** en lugar de CSV porque:
- Ocupa ~5x menos espacio
- Carga ~10x más rápido
- Conserva los tipos de datos

In [ ]:
# Guardar en formato Parquet
df_base.to_parquet(DATA_PROC / 'df_base.parquet', index=False)
df_completo.to_parquet(DATA_PROC / 'df_completo.parquet', index=False)

print('Archivos guardados en data/processed/:')
print(f'  df_base.parquet     → {len(df_base):,} filas (P1, P2)')
print(f'  df_completo.parquet → {len(df_completo):,} filas (P3, P4)')

# Verificar tamaños
import os
for f in ['df_base.parquet', 'df_completo.parquet']:
    size = os.path.getsize(DATA_PROC / f) / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# ─── Crear muestra pequeña para el repo (commit en git) ──────────────────────
sample = df_base.sample(1000, random_state=42)
sample.to_csv('../data/samples/sample_1000.csv', index=False)
print('Muestra de 1000 filas guardada en data/samples/sample_1000.csv')
print('Este archivo SÍ va al repositorio git.')

## 8. Resumen final

| Dataset | Filas | Para |  
|---------|-------|------|
| `df_base` | ~194k | P1 Clustering, P2 Regresión |
| `df_completo` | ~100-120k | P3 Clasificación tipo inst., P4 Migración |

**Notas importantes:**
- El decil de ingreso es **autodeclarado** → sesgo de reporte posible
- Los estudiantes que no se matricularon (~74k) quedan fuera del análisis de P3/P4
- La variable `migra` compara nombres de región — verificar consistencia de strings

**Siguiente paso:** ejecutar `01_preprocesamiento.ipynb`